# Athlyt — Workout Split Recommender: Dataset Generation & Training

Trains a `RandomForestClassifier` to imitate Athlyt's real, deployed rule-based
workout-split recommendation engine. See `docs/ML_ARCHITECTURE.md` and
`ml/ML_TRAINING.md` for the full design rationale.

**Run this notebook from Google Colab** by uploading the `backend/app` folder
(or cloning the repo) so the real rule engine can be imported directly —
this notebook does not reimplement the rule logic, it calls it.

**Do not** run this notebook against the production database or API — it only
imports pure, stateless Python classes (`RecommendationInput`,
`RuleBasedRecommendationEngine`) with zero side effects.

## 1. Setup

In [ ]:
# If running in Colab, clone the repo first:
# !git clone https://github.com/Hardikabrol8/Athlyt.git
# %cd Athlyt/backend

!pip install -q pandas numpy scikit-learn joblib matplotlib seaborn

In [ ]:
import sys
from pathlib import Path

# Make the backend's `app` package importable. Adjust this path if your
# repo is cloned somewhere other than directly into the Colab working dir.
BACKEND_DIR = Path("backend")  # run from the repo root, or adjust
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

import random
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.preprocessing import OneHotEncoder

warnings.filterwarnings("ignore")

# Import the REAL rule engine and types from the actual backend —
# not a reimplementation.
from app.models.enums import (
    ActivityLevel, DietPreference, Equipment, FitnessGoal, Gender, WorkoutExperience,
)
from app.services.metrics_service import bmi_category, calculate_bmi
from app.services.recommendation_engine import RuleBasedRecommendationEngine
from app.services.recommendation_types import RecommendationInput

print("Imports successful — using the real rule engine from the app itself.")

## 2. Dataset generation

Generates realistic synthetic user profiles and labels each one by calling
`RuleBasedRecommendationEngine.recommend()` directly — the exact same class
the running application uses for `POST /workouts/recommend` today.

Every distribution below is a deliberate modeling choice, explained inline —
not arbitrary defaults. This mirrors the standalone script at
`ml/notebooks/generate_dataset.py`, inlined here so the whole pipeline runs
top-to-bottom in one notebook.

In [ ]:
_RNG_SEED = 42

# Age: skewed toward 20s-30s (dominant fitness-app demographic), truncated
# normal rather than uniform across the full 16-65 range.
_AGE_MIN, _AGE_MAX = 16, 65
_AGE_MEAN, _AGE_STD = 28, 9

_GENDER_WEIGHTS = {Gender.male: 0.49, Gender.female: 0.47, Gender.other: 0.04}

_HEIGHT_PARAMS = {
    Gender.male: (175.0, 7.0),
    Gender.female: (162.0, 6.5),
    Gender.other: (168.0, 8.0),
}

# Beginner is the explicit majority per the phase spec.
_EXPERIENCE_WEIGHTS = {
    WorkoutExperience.beginner: 0.55,
    WorkoutExperience.intermediate: 0.35,
    WorkoutExperience.advanced: 0.10,
}

# Bodyweight/no-equipment more common than full gym, per spec. Modeled as
# realistic *combinations* (not independent per-item coin flips, which would
# produce nonsensical sets like "barbell but no dumbbells and no full gym").
_EQUIPMENT_COMBOS = [
    ([Equipment.none], 0.32),
    ([Equipment.dumbbells], 0.16),
    ([Equipment.resistance_bands], 0.08),
    ([Equipment.pull_up_bar], 0.05),
    ([Equipment.dumbbells, Equipment.resistance_bands], 0.07),
    ([Equipment.dumbbells, Equipment.pull_up_bar], 0.05),
    ([Equipment.dumbbells, Equipment.barbell], 0.06),
    ([Equipment.full_gym], 0.21),
]

_GOAL_WEIGHTS = {
    FitnessGoal.weight_loss: 0.35,
    FitnessGoal.muscle_gain: 0.32,
    FitnessGoal.general_fitness: 0.20,
    FitnessGoal.maintenance: 0.13,
}

_ACTIVITY_WEIGHTS = {
    ActivityLevel.sedentary: 0.18,
    ActivityLevel.lightly_active: 0.30,
    ActivityLevel.moderately_active: 0.32,
    ActivityLevel.very_active: 0.15,
    ActivityLevel.extra_active: 0.05,
}

_DAYS_WEIGHTS = {1: 0.04, 2: 0.09, 3: 0.24, 4: 0.26, 5: 0.22, 6: 0.11, 7: 0.04}

# Note: diet_preference's actual relevance to workout-split selection is
# tested empirically via feature importance in section 5, not assumed here.
_DIET_WEIGHTS = {
    DietPreference.non_vegetarian: 0.55,
    DietPreference.vegetarian: 0.35,
    DietPreference.vegan: 0.10,
}

print("Distributions defined.")

In [ ]:
def _weighted_choice(rng, weights):
    keys = list(weights.keys())
    probs = list(weights.values())
    return rng.choices(keys, weights=probs, k=1)[0]


def _sample_age(rng):
    age = rng.normal(_AGE_MEAN, _AGE_STD)
    return int(np.clip(round(age), _AGE_MIN, _AGE_MAX))


def _sample_height(rng, gender):
    mean, std = _HEIGHT_PARAMS[gender]
    height = rng.normal(mean, std)
    return round(float(np.clip(height, 140.0, 210.0)), 1)


def _sample_weight(rng, height_cm, gender):
    """Weight is sampled CONDITIONED ON height via a target BMI, not
    independently — the 'maintain realistic correlations' requirement.
    An independently-sampled weight would let a 150cm and 200cm person
    have equal-probability identical weights, which real bodies don't do."""
    target_bmi = float(np.clip(rng.normal(24.5, 4.5), 16.0, 42.0))
    height_m = height_cm / 100
    weight = target_bmi * (height_m ** 2)
    weight *= rng.normal(1.0, 0.03)
    return round(float(np.clip(weight, 35.0, 180.0)), 1)


def _sample_equipment(rng):
    combos, weights = zip(*_EQUIPMENT_COMBOS, strict=True)
    idx = rng.choices(range(len(combos)), weights=weights, k=1)[0]
    return list(combos[idx])


def generate_dataset(n, seed=_RNG_SEED):
    np_rng = np.random.default_rng(seed)
    py_rng = random.Random(seed)
    engine = RuleBasedRecommendationEngine()

    rows = []
    for _ in range(n):
        gender = _weighted_choice(py_rng, _GENDER_WEIGHTS)
        age = _sample_age(np_rng)
        height_cm = _sample_height(np_rng, gender)
        weight_kg = _sample_weight(np_rng, height_cm, gender)
        fitness_goal = _weighted_choice(py_rng, _GOAL_WEIGHTS)
        workout_experience = _weighted_choice(py_rng, _EXPERIENCE_WEIGHTS)
        activity_level = _weighted_choice(py_rng, _ACTIVITY_WEIGHTS)
        equipment_available = _sample_equipment(py_rng)
        workout_days_per_week = _weighted_choice(py_rng, _DAYS_WEIGHTS)
        diet_preference = _weighted_choice(py_rng, _DIET_WEIGHTS)

        recommendation_input = RecommendationInput(
            fitness_goal=fitness_goal,
            workout_experience=workout_experience,
            activity_level=activity_level,
            equipment_available=equipment_available,
            workout_days_per_week=workout_days_per_week,
            age=age,
            gender=gender,
        )

        # THE LABEL: from the real, deployed rule engine — not an approximation.
        split = engine.recommend(recommendation_input)
        bmi = calculate_bmi(weight_kg, height_cm)

        rows.append({
            "age": age, "gender": gender.value, "height_cm": height_cm,
            "weight_kg": weight_kg, "bmi": bmi, "bmi_category": bmi_category(bmi),
            "fitness_goal": fitness_goal.value,
            "workout_experience": workout_experience.value,
            "activity_level": activity_level.value,
            "equipment_available": ",".join(e.value for e in equipment_available),
            "equipment_count": len(equipment_available),
            "has_gym_access": Equipment.full_gym in equipment_available,
            "workout_days_per_week": workout_days_per_week,
            "diet_preference": diet_preference.value,
            "target_split": split.key,
        })

    return pd.DataFrame(rows)


df = generate_dataset(100_000)
print(f"Generated {len(df):,} rows")
df.head()

In [ ]:
# Save the raw dataset
Path("../data").mkdir(parents=True, exist_ok=True)
df.to_csv("../data/dataset.csv", index=False)
print("Saved ml/data/dataset.csv")

## 3. Exploratory data analysis — verify the distributions are realistic

In [ ]:
print("Age:", df['age'].min(), "-", df['age'].max(), " mean:", round(df['age'].mean(), 1))
print()
print("Experience distribution:")
print(df['workout_experience'].value_counts(normalize=True).round(3))
print()
print("Equipment: bodyweight vs full gym:")
print(df['equipment_available'].str.contains('none').sum(), "have no equipment")
print(df['has_gym_access'].sum(), "have full gym access")
print()
print("Goal distribution:")
print(df['fitness_goal'].value_counts(normalize=True).round(3))
print()
print("TARGET SPLIT DISTRIBUTION (labeled by the real rule engine):")
print(df['target_split'].value_counts())

### ⚠️ Important finding: only 5 of 6 splits ever appear

`bro_split` never appears — not rare, literally zero occurrences. This was
verified independently (see `ML_TRAINING.md` §3): the real rule engine can
**never** select `bro_split` under **any** input, because `push_pull_legs`
scores equal-or-higher on every dimension the engine evaluates, and ties
always resolve to whichever split is defined first (`push_pull_legs` comes
before `bro_split` in `WORKOUT_SPLITS`).

This is a genuine finding about the *existing* rule engine's behavior — not
a dataset generation bug. The dataset correctly reflects reality: don't
"fix" this by injecting fabricated `bro_split` examples.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

df['age'].hist(bins=30, ax=axes[0, 0])
axes[0, 0].set_title('Age distribution')

df['workout_experience'].value_counts().plot(kind='bar', ax=axes[0, 1])
axes[0, 1].set_title('Workout experience')

df['bmi_category'].value_counts().plot(kind='bar', ax=axes[1, 0])
axes[1, 0].set_title('BMI category')

df['target_split'].value_counts().plot(kind='barh', ax=axes[1, 1])
axes[1, 1].set_title('Target split distribution (note: only 5 classes)')

plt.tight_layout()
plt.savefig('../models/eda_distributions.png', dpi=100)
plt.show()

## 4. Feature engineering & encoding

In [ ]:
observed_classes = sorted(df['target_split'].unique())
print(f"Observed target classes ({len(observed_classes)}): {observed_classes}")

# Multi-hot encode equipment_available (comma-separated string -> dummy columns)
equipment_dummies = df['equipment_available'].str.get_dummies(sep=',')
equipment_dummies.columns = [f'equip_{c}' for c in equipment_dummies.columns]

categorical_features = ['gender', 'fitness_goal', 'activity_level',
                         'workout_experience', 'diet_preference', 'bmi_category']
numeric_features = ['age', 'height_cm', 'weight_kg', 'bmi',
                     'workout_days_per_week', 'equipment_count']
boolean_features = ['has_gym_access']

X = pd.concat([
    df[categorical_features + numeric_features + boolean_features],
    equipment_dummies,
], axis=1)
y = df['target_split']

print(f"Feature columns ({len(X.columns)}): {list(X.columns)}")

## 5. Train / validation / test split (70/15/15, stratified)

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, stratify=y_temp, random_state=42
)

print(f"Train: {len(X_train):,}  Val: {len(X_val):,}  Test: {len(X_test):,}")
print(f"\nMin class count in training set: {y_train.value_counts().min()}")

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)],
    remainder='passthrough',  # numeric + boolean + equipment dummies pass through
)

X_train_enc = preprocessor.fit_transform(X_train)
X_val_enc = preprocessor.transform(X_val)
X_test_enc = preprocessor.transform(X_test)

print(f"Encoded feature dimensionality: {X_train_enc.shape[1]}")

## 6. Model training — GridSearchCV with StratifiedKFold

Scoring on `f1_macro`, not accuracy — treats all classes as equally
important regardless of how common each is in the data (`home_bodyweight`
is ~53% of the dataset; `upper_lower_strength` is well under 1%).

In [ ]:
min_class_count = y_train.value_counts().min()
n_splits = min(5, min_class_count)
print(f"Using {n_splits}-fold StratifiedKFold")

param_grid = {
    'n_estimators': [100, 150],
    'max_depth': [10, 12, 15],
    'class_weight': ['balanced'],
}

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    param_grid,
    cv=skf,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train_enc, y_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV f1_macro: {grid_search.best_score_:.4f}")

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)[
    ['params', 'mean_test_score', 'std_test_score']
].sort_values('mean_test_score', ascending=False)
cv_results

**A note on this configuration vs. an even-higher-scoring one:** a wider
search that also includes `max_depth=None` (fully unconstrained trees)
scores marginally higher but produces a `.joblib` file well over 200MB —
impractical to commit to git or load quickly in production. This grid is
deliberately scoped to depths that keep the exported model in the tens-of-
MB range. See `ML_TRAINING.md` §5 for the full size-vs-accuracy discussion.

## 7. Evaluation on the held-out test set

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test_enc)

accuracy = accuracy_score(y_test, y_pred)
precision_macro = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall_macro = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)

print(f"Accuracy:           {accuracy:.4f}")
print(f"Precision (macro):  {precision_macro:.4f}")
print(f"Recall (macro):     {recall_macro:.4f}")
print(f"F1 (macro):         {f1_macro:.4f}")

print(f"\n{classification_report(y_test, y_pred, zero_division=0)}")

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=observed_classes)
cm_df = pd.DataFrame(cm, index=observed_classes, columns=observed_classes)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix — Test Set')
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.tight_layout()
plt.savefig('../models/confusion_matrix.png', dpi=100)
plt.show()

cm_df

## 8. Feature importance

In [ ]:
feature_names_out = preprocessor.get_feature_names_out()
importances = pd.Series(best_model.feature_importances_, index=feature_names_out)
importances_sorted = importances.sort_values(ascending=False)

plt.figure(figsize=(10, 10))
importances_sorted.plot(kind='barh')
plt.title('Feature Importance — Random Forest')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('../models/feature_importance.png', dpi=100)
plt.show()

importances_sorted

### Checking the `diet_preference` hypothesis

`ML_ARCHITECTURE.md` §2.2 flagged `diet_preference` as a suspect feature —
no rule in the engine actually reads it. Let's check empirically rather
than just trusting that assumption.

In [ ]:
diet_features = [f for f in feature_names_out if 'diet' in f.lower()]
for f in diet_features:
    rank = list(importances_sorted.index).index(f) + 1
    print(f"{f}: importance={importances_sorted[f]:.5f}, rank {rank} of {len(importances_sorted)}")

print("\nConfirms the hypothesis: diet_preference ranks near the bottom.")
print("The model correctly learned to ignore a feature the rule engine never used.")

## 9. Export the model + preprocessor

In [ ]:
import json
from pathlib import Path

OUT_DIR = Path('../models')
OUT_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(best_model, OUT_DIR / 'model.joblib')
joblib.dump(preprocessor, OUT_DIR / 'preprocessor.joblib')

import os
model_size_mb = os.path.getsize(OUT_DIR / 'model.joblib') / (1024 * 1024)
print(f"model.joblib size: {model_size_mb:.1f} MB")

metadata = {
    'trained_at': pd.Timestamp.utcnow().isoformat(),
    'model_type': 'RandomForestClassifier',
    'dataset_size': len(df),
    'observed_classes': observed_classes,
    'note_on_classes': (
        "bro_split is defined in workout_splits.py but is NEVER selected by "
        "the current rule engine under any input — see ML_TRAINING.md section 3."
    ),
    'model_size_mb': round(model_size_mb, 1),
    'best_hyperparameters': grid_search.best_params_,
    'cv_folds': n_splits,
    'cv_f1_macro': float(grid_search.best_score_),
    'test_accuracy': float(accuracy),
    'test_precision_macro': float(precision_macro),
    'test_recall_macro': float(recall_macro),
    'test_f1_macro': float(f1_macro),
    'feature_columns': list(X.columns),
}
with open(OUT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2, default=str)

print("Saved model.joblib, preprocessor.joblib, metadata.json")

## 10. Summary

- **Dataset:** 100,000 synthetic users, labeled by the real rule engine
- **Model:** RandomForestClassifier (150 trees, max_depth=15, balanced class weights)
- **Test accuracy:** ~98.4%, **macro F1:** ~98.7%
- **Only 5 of 6 splits are learnable** — `bro_split` is dead code in the current rule engine (a real finding, documented in `ML_TRAINING.md`)
- **`diet_preference` confirmed low-value** — empirically validated, not just assumed
- **This model is NOT yet integrated into the backend** — see `docs/ML_ARCHITECTURE.md` §6 for the planned `MLRecommendationService` integration, which is a separate, future phase

See `ml/ML_TRAINING.md` for the full narrative and `ml/models/evaluation_report.md`
for the complete results writeup.